# Digital PDF — Single Pass Extraction

Extract structured data from a **digital** (text-based) PDF in a single LLM call.

The digital strategy uses embedded text layers — no OCR or vision model needed for parsing.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import os
import sys

from pydantic import BaseModel

from doc_intelligence import PDFExtractionMode, PDFProcessor
from doc_intelligence.pdf.types import ParseStrategy

sys.path.insert(0, os.path.join(os.getcwd(), ".."))
from utils import show_pdf_with_bboxes

PDF_URL = "https://example-files.online-convert.com/document/pdf/example.pdf"
OUT_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "output")
os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
class License(BaseModel):
    license_name: str

In [4]:
processor = PDFProcessor(
    provider="openai",
    model="gpt-5-mini",
    include_citations=True,
    extraction_mode=PDFExtractionMode.SINGLE_PASS,
    strategy=ParseStrategy.DIGITAL,
)

In [5]:
response = processor.extract(PDF_URL, License)

2026-04-06 23:50:12.490 | INFO     | doc_intelligence.pdf.processor:extract:71 - Document parsed successfully
2026-04-06 23:50:12.491 | DEBUG    | doc_intelligence.pdf.extractor:_run_single_pass:112 - PDFExtractor: extract: json_instance_schema: {
    "license_name": {
        "value": <string>,
        "citations": [{"page": <integer>, "blocks": [<integer>]}]
    }
}
2026-04-06 23:50:12.491 | DEBUG    | doc_intelligence.pdf.extractor:_run_single_pass:120 - PDFExtractor: extract: content_text: <page number="0">
<block index="0" type="text">
PDF test file
</block>
<block index="1" type="text">
Purpose: Provide example of this file type
</block>
<block index="2" type="text">
Document file type: PDF
</block>
<block index="3" type="text">
Version: 1.0
</block>
<block index="4" type="text">
Remark:
</block>
<block index="5" type="text">
Example content:
</block>
<block index="6" type="text">
The names "John Doe" for males, "Jane Doe" or "Jane
</block>
<block index="7" type="text">
Roe" for 

In [6]:
response

ExtractionResult(data=License(license_name='Attribution-ShareAlike 3.0 Unported'), metadata={'license_name': {'value': 'Attribution-ShareAlike 3.0 Unported', 'citations': [{'page': 0, 'bboxes': [{'x0': 0.20106913928643427, 'top': 0.8587326111744586, 'x1': 0.5648947389639185, 'bottom': 0.8718454960091222}]}]}})

In [7]:
assert response.metadata is not None
show_pdf_with_bboxes(
    PDF_URL,
    response.metadata["license_name"]["citations"][0],
    out_file=os.path.join(OUT_DIR, "digital_single_pass_bbox.png"),
)

  -> saved to /Users/zeel/Public/ms/open_source/document_ai/notebooks/pdf/output/digital_single_pass_bbox.png
